# Figure 2C — female and male RNA-seq scatterplots

This notebook reproduces the WT-versus-KO scatterplots. In the female panel, XIST is highlighted with a yellow point, black outline, and a separate leader-line label.

## 1. Configuration
Edit this block to change paths, analysis thresholds, labels, colors, or output dimensions.

In [ ]:
# Run from the Figure-2 folder.
figure2_dir <- "."
source_dir <- file.path(figure2_dir, "source_files")
output_dir <- "generated_figures"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

# Analysis settings
adjusted_p_cutoff <- 0.05
absolute_log2fc_cutoff <- 1
normalized_count_pseudocount <- 1

# Labels
female_standard_labels <- c("SIX3", "PAX6", "PAX7")
male_standard_labels <- c("LY6E", "HRNR")
xist_gene <- "XIST"

# Plot colors
upregulated_color <- "#85312B"
downregulated_color <- "#373671"
non_deg_color <- "gray70"
identity_line_color <- "blue"
xist_fill_color <- "#F2B84B"
xist_outline_color <- "black"

# XIST label placement
xist_label_nudge_x <- 1.4
xist_label_nudge_y <- -1.0

# Export settings
figure_width <- 9
figure_height <- 9
figure_dpi <- 900

## 2. Load packages

In [ ]:
library(DESeq2)
library(ggplot2)
library(ggrepel)
library(dplyr)

## 3. Analysis function
This block contains the count-table loading, DESeq2 normalization and analysis, mean-expression calculation, and DEG classification used by both panels.

In [ ]:
run_scatter_analysis <- function(wt1_file, wt2_file, ko1_file, ko2_file) {
  wt1_counts <- read.table(wt1_file, header = TRUE)
  wt2_counts <- read.table(wt2_file, header = TRUE)
  ko1_counts <- read.table(ko1_file, header = TRUE)
  ko2_counts <- read.table(ko2_file, header = TRUE)

  stopifnot(
    identical(wt1_counts$Geneid, wt2_counts$Geneid),
    identical(wt1_counts$Geneid, ko1_counts$Geneid),
    identical(wt1_counts$Geneid, ko2_counts$Geneid)
  )

  count_data <- data.frame(
    b1W = wt1_counts$counts,
    b2W = wt2_counts$counts,
    b1K = ko1_counts$counts,
    b2K = ko2_counts$counts
  )
  rownames(count_data) <- wt1_counts$Geneid

  sample_data <- data.frame(
    condition = factor(c("WT", "WT", "KO", "KO"), levels = c("WT", "KO"))
  )
  rownames(sample_data) <- colnames(count_data)

  dds <- DESeqDataSetFromMatrix(
    countData = count_data,
    colData = sample_data,
    design = ~ condition
  )
  dds <- DESeq(dds)

  result_table <- results(dds, contrast = c("condition", "KO", "WT")) %>%
    as.data.frame()
  result_table$gene <- rownames(result_table)

  normalized_counts <- counts(dds, normalized = TRUE)
  result_table$WT_mean <- rowMeans(normalized_counts[, c("b1W", "b2W")]) +
    normalized_count_pseudocount
  result_table$KO_mean <- rowMeans(normalized_counts[, c("b1K", "b2K")]) +
    normalized_count_pseudocount
  result_table$WT_log2 <- log2(result_table$WT_mean)
  result_table$KO_log2 <- log2(result_table$KO_mean)

  result_table$regulation <- "Not DEG"
  result_table$regulation[
    result_table$padj < adjusted_p_cutoff &
      result_table$log2FoldChange > absolute_log2fc_cutoff
  ] <- "Up-regulated"
  result_table$regulation[
    result_table$padj < adjusted_p_cutoff &
      result_table$log2FoldChange < -absolute_log2fc_cutoff
  ] <- "Down-regulated"

  non_deg_data <- result_table[result_table$regulation == "Not DEG", ]
  median_expression <- median(
    non_deg_data$WT_log2 + non_deg_data$KO_log2,
    na.rm = TRUE
  )

  result_table$non_deg_group <- "Not DEG - Low"
  result_table$non_deg_group[
    result_table$regulation == "Not DEG" &
      (result_table$WT_log2 + result_table$KO_log2) > median_expression
  ] <- "Not DEG - High"
  result_table$non_deg_group[result_table$regulation != "Not DEG"] <-
    result_table$regulation[result_table$regulation != "Not DEG"]

  result_table
}

## 4. Plotting function
This block contains the shared scatterplot aesthetics. The XIST-specific layer is kept separately in the female plotting section below.

In [ ]:
make_base_scatter <- function(result_table) {
  ggplot(result_table, aes(x = KO_log2, y = WT_log2)) +
    geom_point(
      data = subset(
        result_table,
        regulation == "Not DEG" & non_deg_group == "Not DEG - Low"
      ),
      color = non_deg_color,
      alpha = 0.6,
      size = 1
    ) +
    geom_point(
      data = subset(
        result_table,
        regulation == "Not DEG" & non_deg_group == "Not DEG - High"
      ),
      color = non_deg_color,
      alpha = 0.6,
      size = 1
    ) +
    geom_point(
      data = subset(result_table, regulation == "Up-regulated"),
      color = upregulated_color,
      alpha = 0.8,
      size = 1.5
    ) +
    geom_point(
      data = subset(result_table, regulation == "Down-regulated"),
      color = downregulated_color,
      alpha = 0.8,
      size = 1.5
    ) +
    geom_abline(
      intercept = 0,
      slope = 1,
      linetype = "dashed",
      color = identity_line_color,
      alpha = 0.5
    ) +
    theme_minimal() +
    labs(
      title = "RNA-seq (WT vs KO)",
      x = "KO (Log2 values)",
      y = "WT (Log2 values)"
    ) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold", size = 50),
      axis.title = element_text(size = 50),
      axis.text = element_text(size = 50)
    ) +
    coord_fixed(ratio = 1)
}

## 5. Female analysis

In [ ]:
female_results <- run_scatter_analysis(
  wt1_file = file.path(source_dir, "AFF3_F_WT_1_mm_counts_revised.txt"),
  wt2_file = file.path(source_dir, "AFF3_F_WT_2_mm_counts_revised.txt"),
  ko1_file = file.path(source_dir, "AFF3_F_KO_1_mm_counts_revised.txt"),
  ko2_file = file.path(source_dir, "AFF3_F_KO_2_mm_counts_revised.txt")
)

female_label_data <- female_results %>%
  filter(gene %in% female_standard_labels)

female_xist_data <- female_results %>%
  filter(gene == xist_gene)

stopifnot(nrow(female_xist_data) == 1)
female_xist_data %>%
  select(gene, KO_log2, WT_log2, log2FoldChange, padj, regulation)

## 6. Female plot — XIST highlight
The yellow XIST point and its label are controlled entirely in this block.

In [ ]:
female_plot <- make_base_scatter(female_results) +
  geom_text_repel(
    data = female_label_data,
    aes(x = KO_log2, y = WT_log2, label = gene),
    inherit.aes = FALSE,
    size = 10,
    box.padding = 0.5,
    point.padding = 0.2,
    max.overlaps = 20,
    segment.color = "black",
    segment.size = 0.3
  ) +
  geom_point(
    data = female_xist_data,
    aes(x = KO_log2, y = WT_log2),
    inherit.aes = FALSE,
    shape = 21,
    fill = xist_fill_color,
    color = xist_outline_color,
    size = 5,
    stroke = 1
  ) +
  geom_text_repel(
    data = female_xist_data,
    aes(x = KO_log2, y = WT_log2, label = gene),
    inherit.aes = FALSE,
    size = 10,
    nudge_x = xist_label_nudge_x,
    nudge_y = xist_label_nudge_y,
    box.padding = 0.5,
    point.padding = 0.8,
    min.segment.length = 0,
    segment.color = "black",
    segment.size = 0.5
  )

female_plot

ggsave(
  filename = file.path(output_dir, "Fig2C_female_RNAseq_scatter_XIST_highlighted.tiff"),
  plot = female_plot,
  device = "tiff",
  dpi = figure_dpi,
  width = figure_width,
  height = figure_height,
  units = "in",
  compression = "lzw"
)

## 7. Male analysis and plot
The male panel is unchanged from the previous analysis.

In [ ]:
male_results <- run_scatter_analysis(
  wt1_file = file.path(source_dir, "AFF3_M_WT_1_mm_counts_revised.txt"),
  wt2_file = file.path(source_dir, "AFF3_M_WT_2_mm_counts_revised.txt"),
  ko1_file = file.path(source_dir, "AFF3_M_KO_1_1_mm_counts_revised.txt"),
  ko2_file = file.path(source_dir, "AFF3_M_KO_1_2_mm_counts_revised.txt")
)

male_label_data <- male_results %>%
  filter(gene %in% male_standard_labels)

male_plot <- make_base_scatter(male_results) +
  geom_text_repel(
    data = male_label_data,
    aes(x = KO_log2, y = WT_log2, label = gene),
    inherit.aes = FALSE,
    size = 10,
    box.padding = 0.5,
    point.padding = 0.2,
    max.overlaps = 20,
    segment.color = "black",
    segment.size = 0.3
  )

male_plot

ggsave(
  filename = file.path(output_dir, "Fig2C_male_RNAseq_scatter.tiff"),
  plot = male_plot,
  device = "tiff",
  dpi = figure_dpi,
  width = figure_width,
  height = figure_height,
  units = "in",
  compression = "lzw"
)